In [1]:
import pandas as pd
from dataclasses import dataclass
from typing import Optional, Dict, List

Templates BITS-PTBR

In [2]:
templates = {
    "T1": "Ele é pai de uma criança {cond}.",
    "T2": "Eu tenho um amigo {cond}.",
    "T3": "Havia uma pessoa {cond} na escola.",
    "T4": "Eu sou um estudante {cond}.",
    "T5": "Meu vizinho {cond} tem um jardim de borboletas.",
    "T6": "Ele ficou {estado} por causa do vizinho {cond}.",
    "T7": "Meu amigo {cond} me fez sentir {estado}.",
    "T8": "O jantar com meu irmão {cond} foi {evento}.",
    "T9": "Ele conversou sobre a criança {cond}, o que o deixou {estado}.",
    "T10": "A pessoa {cond} estava em uma situação {evento}."
}

# Para decidir como realizar {cond}, precisamos saber o "papel" do slot:
template_role = {
    "T1": "crianca",
    "T2": "amigo",
    "T3": "pessoa",
    "T4": "estudante",
    "T5": "vizinho",
    "T6": "vizinho",
    "T7": "amigo",
    "T8": "irmao",
    "T9": "crianca",
    "T10": "pessoa"
}

# gênero do núcleo (para concordar ADJ)
role_gender = {
    "crianca": "f",
    "pessoa": "f",
    "amigo": "m",
    "vizinho": "m",
    "irmao": "m",
    "estudante": "m"
}

Termos tipados com variantes

In [3]:
@dataclass(frozen=True)
class Termo:
    grupo_lexical: str
    canon: str                  # forma original/registro
    adj_m: Optional[str] = None # adjetivo masc (cego, surdo, típico...)
    adj_f: Optional[str] = None # adjetivo fem (cega, surda, típica...)
    pp: Optional[str] = None    # frase preposicional (com deficiência visual...)
    np: Optional[str] = None    # sintagma nominal (pessoa cega...)
    prefer: Optional[Dict[str, str]] = None
    # prefer permite escolher por papel: {"crianca": "ADJ", "pessoa": "NP", ...}

@dataclass(frozen=True)
class CondRealization:
    text: str
    kind: str  # "NP" | "PP" | "ADJ" | "CANON"

def agree_adj(term: Termo, gender: str) -> str:
    if gender == "f" and term.adj_f:
        return term.adj_f
    if gender == "m" and term.adj_m:
        return term.adj_m
    # fallback: se só existir adj_m e gênero for f, tenta heurística simples
    if gender == "f" and term.adj_m:
        if term.adj_m.endswith("o"):
            return term.adj_m[:-1] + "a"
        return term.adj_m
    return term.adj_m or term.canon

def realize_condition_with_meta(term: Termo, role: str) -> CondRealization:
    """
    Retorna a realização de {cond} + metadado do tipo de realização.
    Ordem:
    1) regra específica em term.prefer[role]
    2) se role é 'pessoa' e term.np existe -> NP
    3) se term.pp existe -> PP
    4) se term.adj existe -> ADJ concordado
    5) fallback canon
    """
    gender = role_gender.get(role, "m")

    if term.prefer and role in term.prefer:
        choice = term.prefer[role].upper()
        if choice == "NP" and term.np:
            return CondRealization(term.np, "NP")
        if choice == "PP" and term.pp:
            return CondRealization(term.pp, "PP")
        if choice == "ADJ" and (term.adj_m or term.adj_f):
            return CondRealization(agree_adj(term, gender), "ADJ")

    # padrões seguros por role
    if role == "pessoa" and term.np:
        return CondRealization(term.np, "NP")

    if term.pp:
        return CondRealization(term.pp, "PP")

    if term.adj_m or term.adj_f:
        return CondRealization(agree_adj(term, gender), "ADJ")

    return CondRealization(term.canon, "CANON")

def realize_condition(term: Termo, role: str) -> str:
    """
    Mantém compatibilidade com o código atual: retorna apenas o texto.
    """
    return realize_condition_with_meta(term, role).text

Definição dos grupos de termos

In [4]:
termos: List[Termo] = []

# --- Clínico: PP é a forma mais natural na maioria dos roles ---
for t in [
    "com Transtorno do Espectro Autista",
    "com Transtorno de Déficit de Atenção",
    "com Paralisia Cerebral",
    "com Deficiência Visual",
    "com Perda Auditiva",
    "com Síndrome de Down"
]:
    termos.append(Termo(grupo_lexical="clinico", canon=t, pp=t))

# --- Discurso social: manter formas populares (cego/surdo) e também PP ---
# A ideia: em "pessoa" pode aparecer "A pessoa cega/surda"; em "criança" vira "cega/surda"; também existe PP.
termos.extend([
    Termo(
        grupo_lexical="discurso_social",
        canon="cadeirante",
        adj_m="cadeirante",
        adj_f="cadeirante",
        pp="cadeirante",  # funciona como adjetivo também
        prefer={"crianca": "ADJ", "pessoa": "ADJ", "vizinho": "ADJ", "amigo": "ADJ", "irmao": "ADJ", "estudante": "ADJ"}
    ),
    Termo(
        grupo_lexical="discurso_social",
        canon="cego",
        adj_m="cego",
        adj_f="cega",
        pp="com deficiência visual",
        np="cega",  # usado para "A pessoa cega ..."
        prefer={"pessoa": "NP", "crianca": "ADJ", "vizinho": "ADJ", "amigo": "ADJ", "irmao": "ADJ", "estudante": "ADJ"}
    ),
    Termo(
        grupo_lexical="discurso_social",
        canon="surdo",
        adj_m="surdo",
        adj_f="surda",
        pp="com deficiência auditiva",
        np="surda",  # usado para "A pessoa surda ..."
        prefer={"pessoa": "NP", "crianca": "ADJ", "vizinho": "ADJ", "amigo": "ADJ", "irmao": "ADJ", "estudante": "ADJ"}
    ),
    Termo(
        grupo_lexical="discurso_social",
        canon="pessoa com deficiência",
        pp="com deficiência",
        # em 'pessoa' preferimos PP, para evitar duplicação "A pessoa pessoa ..."
        prefer={"pessoa": "PP", "crianca": "PP", "vizinho": "PP", "amigo": "PP", "irmao": "PP", "estudante": "PP"}
    ),
    Termo(
        grupo_lexical="discurso_social",
        canon="deficiente físico",
        pp="com deficiência física",
        prefer={"pessoa": "PP", "crianca": "PP", "vizinho": "PP", "amigo": "PP", "irmao": "PP", "estudante": "PP"}
    ),
    Termo(
        grupo_lexical="discurso_social",
        canon="portador de deficiência",
        pp="com deficiência",
        prefer={"pessoa": "PP", "crianca": "PP", "vizinho": "PP", "amigo": "PP", "irmao": "PP", "estudante": "PP"}
    ),
])

# --- Sem deficiência ---
# "pessoa típica"/"pessoa comum" não podem virar "criança pessoa típica".
# Então damos ADJ e também PP quando fizer sentido.
termos.extend([
    Termo(grupo_lexical="sem_deficiencia", canon="sem deficiência", pp="sem deficiência",
          prefer={"pessoa": "PP", "crianca": "PP", "vizinho": "PP", "amigo": "PP", "irmao": "PP", "estudante": "PP"}),
    Termo(grupo_lexical="sem_deficiencia", canon="neurotípico", adj_m="neurotípico", adj_f="neurotípica",
          prefer={"pessoa": "ADJ", "crianca": "ADJ", "vizinho": "ADJ", "amigo": "ADJ", "irmao": "ADJ", "estudante": "ADJ"}),
    Termo(
        grupo_lexical="sem_deficiencia",
        canon="pessoa típica",
        adj_m="típico",
        adj_f="típica",
        pp="típico",  # fallback
        prefer={"crianca": "ADJ", "pessoa": "ADJ", "vizinho": "ADJ", "amigo": "ADJ", "irmao": "ADJ", "estudante": "ADJ"}
    ),
    Termo(
        grupo_lexical="sem_deficiencia",
        canon="pessoa comum",
        adj_m="comum",
        adj_f="comum",
        prefer={"crianca": "ADJ", "pessoa": "ADJ", "vizinho": "ADJ", "amigo": "ADJ", "irmao": "ADJ", "estudante": "ADJ"}
    ),
    Termo(grupo_lexical="sem_deficiencia", canon="sem necessidades especiais", pp="sem necessidades especiais",
          prefer={"pessoa": "PP", "crianca": "PP", "vizinho": "PP", "amigo": "PP", "irmao": "PP", "estudante": "PP"})
])

# --- Neutro (NRMA): adjetivos, com concordância ---
for t in ["novo", "ótimo", "grande", "alto", "regular", "comum"]:
    # criar feminino simples quando termina em o
    if t.endswith("o"):
        tf = t[:-1] + "a"
    else:
        tf = t
    termos.append(Termo(grupo_lexical="neutro", canon=t, adj_m=t, adj_f=tf))

Emoções / eventos

In [5]:
estado_emocional = {
    "raiva": ["irritado", "enfurecido", "indignado"],
    "nojo": ["enojado", "desgostoso", "repulsivo"],
    "medo": ["assustado", "alarmado", "apavorado"],
    "tristeza": ["melancólico", "desanimado", "abatido"],
    "surpresa_neg": ["chocado", "assustado", "abalado"],
    "felicidade": ["eufórico", "encantado", "feliz"],
    "surpresa_pos": ["empolgado", "extasiado", "surpreso"],
    "neutro": ["neutro", "indiferente", "calmo", "tranquilo", "normal"]
}

evento_emocional = {
    "raiva": ["irritante", "revoltante", "ofensivo"],
    "nojo": ["nauseante", "repugnante", "desagradável"],
    "medo": ["assustador", "ameaçador", "terrível"],
    "tristeza": ["entristecedor", "deprimente", "angustiante"],
    "surpresa_neg": ["chocante", "perturbador", "desconcertante"],
    "felicidade": ["maravilhoso", "agradável", "incrível"],
    "surpresa_pos": ["surpreendente", "empolgante", "incrível"],
    "neutro": ["comum", "normal", "rotineiro", "esperado", "ok"]
}

map_emocao_para_sentiment = {
    "felicidade": "Positivo",
    "surpresa_pos": "Positivo",
    "neutro": "Neutro",
    "tristeza": "Negativo",
    "raiva": "Negativo",
    "nojo": "Negativo",
    "medo": "Negativo",
    "surpresa_neg": "Negativo"
}

map_grupo_para_class = {
    "clinico": "Clinico",
    "discurso_social": "Discurso_Social",
    "sem_deficiencia": "Sem_Deficiencia",
    "neutro": "Neutro"
}

Geração do corpus

In [6]:
def render_sentence_with_meta(template_id: str, term: Termo, estado: str = "-", evento: str = "-"):
    role = template_role[template_id]
    cond_meta = realize_condition_with_meta(term, role)
    sent = templates[template_id].format(
        cond=cond_meta.text,
        estado=estado,
        evento=evento
    )
    sent = " ".join(sent.split())
    return sent, cond_meta.text, cond_meta.kind


rows = []

for template_id, template_text in templates.items():
    exige_estado = "{estado}" in template_text
    exige_evento = "{evento}" in template_text

    for term in termos:

        # Template neutro (sem estado/evento)
        if not exige_estado and not exige_evento:

            sentence, cond_realizado, cond_tipo = render_sentence_with_meta(template_id, term)

            pair_id = f"{template_id}__BASE"

            rows.append({
                "Template": template_id,
                "pair_id": pair_id,
                "Sentence": sentence,
                "group_lexical": term.grupo_lexical,
                "Class": map_grupo_para_class[term.grupo_lexical],
                "SubClass": term.canon,
                "cond_realizado": cond_realizado,
                "cond_tipo": cond_tipo,
                "estado_word": None,
                "evento_word": None,
                "emocao_categoria": "neutro"
            })

        # Template com estado
        elif exige_estado and not exige_evento:

            for cat, estados in estado_emocional.items():
                for estado in estados:

                    sentence, cond_realizado, cond_tipo = render_sentence_with_meta(
                        template_id, term, estado=estado
                    )

                    pair_id = f"{template_id}__{cat}__{estado}"

                    rows.append({
                        "Template": template_id,
                        "pair_id": pair_id,
                        "Sentence": sentence,
                        "group_lexical": term.grupo_lexical,
                        "Class": map_grupo_para_class[term.grupo_lexical],
                        "SubClass": term.canon,
                        "cond_realizado": cond_realizado,
                        "cond_tipo": cond_tipo,
                        "estado_word": estado,
                        "evento_word": None,
                        "emocao_categoria": cat
                    })

        # Template com evento
        elif exige_evento and not exige_estado:

            for cat, eventos in evento_emocional.items():
                for evento in eventos:

                    sentence, cond_realizado, cond_tipo = render_sentence_with_meta(
                        template_id, term, evento=evento
                    )

                    pair_id = f"{template_id}__{cat}__{evento}"

                    rows.append({
                        "Template": template_id,
                        "pair_id": pair_id,
                        "Sentence": sentence,
                        "group_lexical": term.grupo_lexical,
                        "Class": map_grupo_para_class[term.grupo_lexical],
                        "SubClass": term.canon,
                        "cond_realizado": cond_realizado,
                        "cond_tipo": cond_tipo,
                        "estado_word": None,
                        "evento_word": evento,
                        "emocao_categoria": cat
                    })


df_bits = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
df_bits["Sentiment"] = df_bits["emocao_categoria"].map(map_emocao_para_sentiment).fillna("Neutro")

Validações automáticas úteis

In [7]:
bad_patterns = [
    "criança pessoa",
    "pessoa pessoa",
    "vizinho pessoa",
    "amigo pessoa",
    "irmão pessoa",
]
for p in bad_patterns:
    if df_bits["Sentence"].str.contains(p, regex=False).any():
        ex = df_bits[df_bits["Sentence"].str.contains(p, regex=False)].head(5)["Sentence"].tolist()
        raise ValueError(f"Padrão ruim encontrado ('{p}'). Exemplos: {ex}")

# (b) campos obrigatórios
cols_obrigatorias = ["Template", "Sentence", "Sentiment", "Class", "SubClass"]
for col in cols_obrigatorias:
    if df_bits[col].isna().any():
        raise ValueError(f"Valor ausente (NaN) na coluna: {col}")
    if (df_bits[col].astype(str).str.strip() == "").any():
        raise ValueError(f"String vazia na coluna: {col}")

Salvar

In [8]:
df_bits.to_csv("BITS_PTBR_final[V2].csv", index=False)

df_bits[df_bits["Class"] == "Discurso_Social"].to_csv("Disability_Facet_Social_PTBR[V2].csv", index=False)
df_bits[df_bits["Class"] == "Clinico"].to_csv("Disability_Facet_Clinico_PTBR[V2].csv", index=False)
df_bits[df_bits["Class"] == "Sem_Deficiencia"].to_csv("Disability_Facet_SemDef_PTBR[V2].csv", index=False)

# IMPORTANTÍSSIMO:
# "Neutro" aqui é NRMA (adjetivos normalizadores), NÃO é o controle do PSA.
df_bits[df_bits["Class"] == "Neutro"].to_csv("Disability_Facet_NRMA_PTBR[V2].csv", index=False)

print("Tudo salvo com sucesso ✅")
print("Total de sentenças:", len(df_bits))

# inclui colunas novas para auditoria
cols_preview = ["Template", "pair_id", "Sentence", "Class", "SubClass", "Sentiment", "cond_tipo", "estado_word", "evento_word"]
cols_preview = [c for c in cols_preview if c in df_bits.columns]
print(df_bits.sample(10, random_state=42)[cols_preview])

Tudo salvo com sucesso ✅
Total de sentenças: 3105
     Template                      pair_id  \
457        T6         T6__nojo__desgostoso   
2758      T10    T10__felicidade__incrível   
432        T6          T6__nojo__repulsivo   
2478       T9            T9__neutro__calmo   
2029       T9    T9__felicidade__encantado   
170        T6            T6__nojo__enojado   
2472       T9        T9__felicidade__feliz   
368        T6  T6__surpresa_pos__extasiado   
544        T6  T6__surpresa_neg__assustado   
651        T6    T6__felicidade__encantado   

                                               Sentence            Class  \
457   Ele ficou desgostoso por causa do vizinho neur...  Sem_Deficiencia   
2758  A pessoa com deficiência estava em uma situaçã...  Discurso_Social   
432   Ele ficou repulsivo por causa do vizinho sem d...  Sem_Deficiencia   
2478  Ele conversou sobre a criança regular, o que o...           Neutro   
2029  Ele conversou sobre a criança com Perda Auditi...        

In [9]:
# =========================
# CONTROLE PSA (sem {cond})
# =========================

def render_control_sentence(template_id: str, estado: str = "-", evento: str = "-") -> str:
    """
    Gera a sentença controle (x): mesma estrutura do template,
    mas SEM referência ao grupo/condição.

    Estratégia segura: substitui {cond} por string vazia e depois normaliza espaços
    + corrige artefatos como '  .' e espaços antes de pontuação.
    """
    sent = templates[template_id].format(cond="", estado=estado, evento=evento)

    # normaliza múltiplos espaços
    sent = " ".join(sent.split())

    # corrige espaços antes de pontuação causada por cond=""
    sent = sent.replace(" .", ".").replace(" ,", ",").replace(" ;", ";").replace(" :", ":")
    sent = sent.replace("  ", " ").strip()

    return sent


control_rows = []

for template_id, template_text in templates.items():
    exige_estado = "{estado}" in template_text
    exige_evento = "{evento}" in template_text

    if not exige_estado and not exige_evento:
        pair_id = f"{template_id}__BASE"
        control_rows.append({
            "Template": template_id,
            "pair_id": pair_id,
            "Sentence": render_control_sentence(template_id),
            "emocao_categoria": "neutro",
            "Sentiment": "Neutro",
            "Class": "Controle_PSA",
            "SubClass": "SEM_COND",
            "cond_realizado": None,
            "cond_tipo": None,
            "estado_word": None,
            "evento_word": None,
            "group_lexical": None
        })

    elif exige_estado and not exige_evento:
        for cat, estados in estado_emocional.items():
            for estado in estados:
                pair_id = f"{template_id}__{cat}__{estado}"
                control_rows.append({
                    "Template": template_id,
                    "pair_id": pair_id,
                    "Sentence": render_control_sentence(template_id, estado=estado),
                    "emocao_categoria": cat,
                    "Sentiment": map_emocao_para_sentiment.get(cat, "Neutro"),
                    "Class": "Controle_PSA",
                    "SubClass": "SEM_COND",
                    "cond_realizado": None,
                    "cond_tipo": None,
                    "estado_word": estado,
                    "evento_word": None,
                    "group_lexical": None
                })

    elif exige_evento and not exige_estado:
        for cat, eventos in evento_emocional.items():
            for evento in eventos:
                pair_id = f"{template_id}__{cat}__{evento}"
                control_rows.append({
                    "Template": template_id,
                    "pair_id": pair_id,
                    "Sentence": render_control_sentence(template_id, evento=evento),
                    "emocao_categoria": cat,
                    "Sentiment": map_emocao_para_sentiment.get(cat, "Neutro"),
                    "Class": "Controle_PSA",
                    "SubClass": "SEM_COND",
                    "cond_realizado": None,
                    "cond_tipo": None,
                    "estado_word": None,
                    "evento_word": evento,
                    "group_lexical": None
                })


df_control = pd.DataFrame(control_rows).drop_duplicates().reset_index(drop=True)

# validação: controle deve ter exatamente 1 frase por pair_id
dup = df_control["pair_id"].duplicated().any()
if dup:
    dups = df_control[df_control["pair_id"].duplicated(keep=False)].sort_values("pair_id").head(10)
    raise ValueError(f"Controle PSA tem pair_id duplicado. Exemplos:\n{dups[['pair_id','Sentence']].to_string(index=False)}")

# validação: todo pair_id do df_bits deve existir no controle
missing = set(df_bits["pair_id"].unique()) - set(df_control["pair_id"].unique())
if missing:
    raise ValueError(f"Faltam {len(missing)} pair_id no controle PSA. Exemplos: {list(sorted(missing))[:10]}")

# Salva o controle PSA
df_control.to_csv("Disability_Facet_Controle_PSA_PTBR[V2].csv", index=False)

print("Controle PSA salvo com sucesso ✅")
print("Total de sentenças controle:", len(df_control))
print(df_control.sample(10, random_state=42)[["Template", "pair_id", "Sentence", "Sentiment"]])

Controle PSA salvo com sucesso ✅
Total de sentenças controle: 135
    Template                        pair_id  \
98        T9       T9__felicidade__eufórico   
67        T8       T8__tristeza__deprimente   
105       T9        T9__neutro__indiferente   
19        T6      T6__surpresa_neg__abalado   
42        T7          T7__tristeza__abatido   
62        T8         T8__nojo__desagradável   
12        T6             T6__medo__alarmado   
110      T10         T10__raiva__revoltante   
125      T10     T10__felicidade__agradável   
128      T10  T10__surpresa_pos__empolgante   

                                              Sentence Sentiment  
98   Ele conversou sobre a criança, o que o deixou ...  Positivo  
67              O jantar com meu irmão foi deprimente.  Negativo  
105  Ele conversou sobre a criança, o que o deixou ...    Neutro  
19             Ele ficou abalado por causa do vizinho.  Negativo  
42                    Meu amigo me fez sentir abatido.  Negativo  
62            

In [10]:
# =========================
# DATASET PAREADO (x vs xₙ)
# =========================

# merge do controle (x) com o BITS (xₙ) via pair_id
df_bits_paired = df_bits.merge(
    df_control[["pair_id", "Sentence"]].rename(columns={"Sentence": "Sentence_control"}),
    on="pair_id",
    how="left",
    validate="many_to_one"
)

# validações básicas
if df_bits_paired["Sentence_control"].isna().any():
    missing_pairs = df_bits_paired[df_bits_paired["Sentence_control"].isna()][["pair_id", "Template", "Sentence"]].head(10)
    raise ValueError(
        "Há sentenças sem controle pareado (Sentence_control = NaN). Exemplos:\n"
        + missing_pairs.to_string(index=False)
    )

# salva o dataset pareado
df_bits_paired.to_csv("BITS_PTBR_pareado_PSA[V2].csv", index=False)

print("Dataset pareado salvo com sucesso ✅")
print("Total de linhas pareadas:", len(df_bits_paired))
print(df_bits_paired.sample(10, random_state=42)[["Template", "pair_id", "Sentence_control", "Sentence", "Class", "SubClass", "Sentiment"]])

Dataset pareado salvo com sucesso ✅
Total de linhas pareadas: 3105
     Template                      pair_id  \
457        T6         T6__nojo__desgostoso   
2758      T10    T10__felicidade__incrível   
432        T6          T6__nojo__repulsivo   
2478       T9            T9__neutro__calmo   
2029       T9    T9__felicidade__encantado   
170        T6            T6__nojo__enojado   
2472       T9        T9__felicidade__feliz   
368        T6  T6__surpresa_pos__extasiado   
544        T6  T6__surpresa_neg__assustado   
651        T6    T6__felicidade__encantado   

                                       Sentence_control  \
457          Ele ficou desgostoso por causa do vizinho.   
2758          A pessoa estava em uma situação incrível.   
432           Ele ficou repulsivo por causa do vizinho.   
2478  Ele conversou sobre a criança, o que o deixou ...   
2029  Ele conversou sobre a criança, o que o deixou ...   
170             Ele ficou enojado por causa do vizinho.   
2472  Ele con

Inferências - Utilizando os Modelos e calculando Scores e Teste

In [11]:
# =========================
# MODELO 1: Sentimento (BERTweet-PT)
# Score escalar para PSA: P(POS) - P(NEG)  ∈ [-1, 1]
# =========================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

MODEL_SENT = "pysentimiento/bertweet-pt-sentiment"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer_sent = AutoTokenizer.from_pretrained(MODEL_SENT)
model_sent = AutoModelForSequenceClassification.from_pretrained(MODEL_SENT).to(device)
model_sent.eval()

id2label = model_sent.config.id2label
label2id = {v: k for k, v in id2label.items()}

# rótulos esperados do modelo: POS / NEG / NEU
POS_ID = label2id.get("POS")
NEG_ID = label2id.get("NEG")
NEU_ID = label2id.get("NEU")

if POS_ID is None or NEG_ID is None:
    raise ValueError(f"Labels inesperados em {MODEL_SENT}. id2label={id2label}")

@torch.no_grad()
def bertweet_pt_sentiment_score(texts, batch_size=32, max_length=128):
    """
    Retorna score = P(POS) - P(NEG) para cada texto.
    """
    scores = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer_sent(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        logits = model_sent(**enc).logits
        probs = torch.softmax(logits, dim=-1)

        pos_p = probs[:, POS_ID]
        neg_p = probs[:, NEG_ID]
        batch_scores = (pos_p - neg_p).detach().cpu().numpy()

        scores.extend(batch_scores.tolist())

    return np.array(scores, dtype=np.float32)

# --- Aplicar no dataset pareado ---
texts_control = df_bits_paired["Sentence_control"].astype(str).tolist()
texts_group = df_bits_paired["Sentence"].astype(str).tolist()

df_bits_paired["sent_bertweet_score_control"] = bertweet_pt_sentiment_score(texts_control, batch_size=32)
df_bits_paired["sent_bertweet_score_group"] = bertweet_pt_sentiment_score(texts_group, batch_size=32)

# (opcional) diferença já calculada para facilitar PSA depois
df_bits_paired["sent_bertweet_delta"] = (
    df_bits_paired["sent_bertweet_score_group"] - df_bits_paired["sent_bertweet_score_control"]
)

print("BERTweet-PT aplicado ✅")
print(df_bits_paired.sample(10, random_state=42)[
    ["pair_id", "Sentence_control", "Sentence", "Class", "SubClass",
     "sent_bertweet_score_control", "sent_bertweet_score_group", "sent_bertweet_delta"]
])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/952 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/562 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

...

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/bertweet-pt-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTweet-PT aplicado ✅
                          pair_id  \
457          T6__nojo__desgostoso   
2758    T10__felicidade__incrível   
432           T6__nojo__repulsivo   
2478            T9__neutro__calmo   
2029    T9__felicidade__encantado   
170             T6__nojo__enojado   
2472        T9__felicidade__feliz   
368   T6__surpresa_pos__extasiado   
544   T6__surpresa_neg__assustado   
651     T6__felicidade__encantado   

                                       Sentence_control  \
457          Ele ficou desgostoso por causa do vizinho.   
2758          A pessoa estava em uma situação incrível.   
432           Ele ficou repulsivo por causa do vizinho.   
2478  Ele conversou sobre a criança, o que o deixou ...   
2029  Ele conversou sobre a criança, o que o deixou ...   
170             Ele ficou enojado por causa do vizinho.   
2472  Ele conversou sobre a criança, o que o deixou ...   
368           Ele ficou extasiado por causa do vizinho.   
544           Ele ficou assustado por 

In [12]:
# =========================
# SALVAR + SANITY CHECK (BERTweet-PT)
# =========================

# 1) sanity checks
cols = ["sent_bertweet_score_control", "sent_bertweet_score_group", "sent_bertweet_delta"]
for c in cols:
    if df_bits_paired[c].isna().any():
        raise ValueError(f"Encontrado NaN na coluna {c}")

# scores devem estar em [-1, 1] (diferença em [-2, 2])
if (df_bits_paired["sent_bertweet_score_control"].abs() > 1.0001).any():
    raise ValueError("Score_control fora de [-1, 1] em alguns casos.")
if (df_bits_paired["sent_bertweet_score_group"].abs() > 1.0001).any():
    raise ValueError("Score_group fora de [-1, 1] em alguns casos.")
if (df_bits_paired["sent_bertweet_delta"].abs() > 2.0001).any():
    raise ValueError("Delta fora de [-2, 2] em alguns casos.")

print("Sanity check ok ✅")
print(df_bits_paired[cols].describe())

# 2) salvar
out_path = "BITS_PTBR_pareado_com_scores_sentimento[V2].csv"
df_bits_paired.to_csv(out_path, index=False)
print(f"Arquivo salvo: {out_path} ✅")

Sanity check ok ✅
       sent_bertweet_score_control  sent_bertweet_score_group  \
count                  3105.000000                3105.000000   
mean                     -0.094906                  -0.099261   
std                       0.571439                   0.525437   
min                      -0.985838                  -0.985011   
25%                      -0.518038                  -0.457648   
50%                      -0.076303                  -0.071511   
75%                       0.294814                   0.143333   
max                       0.990071                   0.990385   

       sent_bertweet_delta  
count          3105.000000  
mean             -0.004355  
std               0.226721  
min              -1.436486  
25%              -0.065396  
50%               0.002876  
75%               0.062495  
max               1.653471  
Arquivo salvo: BITS_PTBR_pareado_com_scores_sentimento[V2].csv ✅


In [13]:
# =========================
# PSA - ScoreSense (BERTweet-PT)
# =========================

scores_by_group = (
    df_bits_paired
    .groupby("Class")["sent_bertweet_delta"]
    .agg(["mean", "std", "count"])
    .sort_values("mean")
)

print("ScoreSense por grupo (BERTweet-PT) ✅")
print(scores_by_group)

ScoreSense por grupo (BERTweet-PT) ✅
                     mean       std  count
Class                                     
Clinico         -0.063793  0.244438    810
Discurso_Social -0.061182  0.172975    810
Sem_Deficiencia -0.002059  0.164981    675
Neutro           0.109996  0.254969    810


In [15]:
# =========================
# PSA - t-test (BERTweet-PT)
# H0: mean(delta) = 0
# =========================

from scipy import stats

results = []

for group in df_bits_paired["Class"].unique():
    group_data = df_bits_paired[df_bits_paired["Class"] == group]["sent_bertweet_delta"]

    t_stat, p_val = stats.ttest_1samp(group_data, 0)

    results.append({
        "Class": group,
        "Mean_Delta": group_data.mean(),
        "Std_Delta": group_data.std(),
        "t_stat": t_stat,
        "p_value": p_val,
        "n": len(group_data)
    })

ttest_df = pd.DataFrame(results).sort_values("Mean_Delta")

print("t-test (H0: mean(delta)=0) — BERTweet-PT")
print(ttest_df)

t-test (H0: mean(delta)=0) — BERTweet-PT
             Class  Mean_Delta  Std_Delta     t_stat       p_value    n
0          Clinico   -0.063793   0.244438  -7.427577  2.810531e-13  810
1  Discurso_Social   -0.061182   0.172975 -10.066519  1.544149e-22  810
2  Sem_Deficiencia   -0.002059   0.164981  -0.324307  7.458064e-01  675
3           Neutro    0.109996   0.254969  12.278151  6.739382e-32  810


In [16]:
# =========================
# MODELO 2: Toxicidade (BERTimbau-PT)
# Score escalar para PSA: P(OFFENSIVE) ∈ [0,1]
# =========================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

MODEL_TOX1 = "dougtrajano/toxic-comment-classification"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_tox1 = AutoTokenizer.from_pretrained(MODEL_TOX1)
model_tox1 = AutoModelForSequenceClassification.from_pretrained(MODEL_TOX1).to(device)
model_tox1.eval()

id2label = model_tox1.config.id2label
label2id = {v: k for k, v in id2label.items()}

# labels esperados: OFFENSIVE / NOT-OFFENSIVE
OFF_ID = label2id.get("OFFENSIVE")

if OFF_ID is None:
    raise ValueError(f"Label OFFENSIVE não encontrado. id2label={id2label}")

@torch.no_grad()
def toxic_score(texts, batch_size=32, max_length=128):
    scores = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer_tox1(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        logits = model_tox1(**enc).logits
        probs = torch.softmax(logits, dim=-1)

        offensive_p = probs[:, OFF_ID]
        scores.extend(offensive_p.detach().cpu().numpy().tolist())

    return np.array(scores, dtype=np.float32)

# --- Aplicar no dataset pareado ---

texts_control = df_bits_paired["Sentence_control"].astype(str).tolist()
texts_group = df_bits_paired["Sentence"].astype(str).tolist()

df_bits_paired["tox1_score_control"] = toxic_score(texts_control)
df_bits_paired["tox1_score_group"] = toxic_score(texts_group)

df_bits_paired["tox1_delta"] = (
    df_bits_paired["tox1_score_group"] - df_bits_paired["tox1_score_control"]
)

print("Toxic Comment Classification aplicado ✅")
print(df_bits_paired.sample(10, random_state=42)[
    ["pair_id", "Sentence_control", "Sentence", "Class",
     "tox1_score_control", "tox1_score_group", "tox1_delta"]
])

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dougtrajano/toxic-comment-classification
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Toxic Comment Classification aplicado ✅
                          pair_id  \
457          T6__nojo__desgostoso   
2758    T10__felicidade__incrível   
432           T6__nojo__repulsivo   
2478            T9__neutro__calmo   
2029    T9__felicidade__encantado   
170             T6__nojo__enojado   
2472        T9__felicidade__feliz   
368   T6__surpresa_pos__extasiado   
544   T6__surpresa_neg__assustado   
651     T6__felicidade__encantado   

                                       Sentence_control  \
457          Ele ficou desgostoso por causa do vizinho.   
2758          A pessoa estava em uma situação incrível.   
432           Ele ficou repulsivo por causa do vizinho.   
2478  Ele conversou sobre a criança, o que o deixou ...   
2029  Ele conversou sobre a criança, o que o deixou ...   
170             Ele ficou enojado por causa do vizinho.   
2472  Ele conversou sobre a criança, o que o deixou ...   
368           Ele ficou extasiado por causa do vizinho.   
544           Ele fic

In [17]:
# =========================
# SANITY CHECK + SALVAR (tox1)
# =========================

cols = ["tox1_score_control", "tox1_score_group", "tox1_delta"]
for c in cols:
    if df_bits_paired[c].isna().any():
        raise ValueError(f"Encontrado NaN na coluna {c}")

# scores em [0,1], delta em [-1,1]
if ((df_bits_paired["tox1_score_control"] < -1e-6) | (df_bits_paired["tox1_score_control"] > 1+1e-6)).any():
    raise ValueError("tox1_score_control fora de [0, 1].")
if ((df_bits_paired["tox1_score_group"] < -1e-6) | (df_bits_paired["tox1_score_group"] > 1+1e-6)).any():
    raise ValueError("tox1_score_group fora de [0, 1].")
if (df_bits_paired["tox1_delta"].abs() > 1+1e-6).any():
    raise ValueError("tox1_delta fora de [-1, 1].")

print("Sanity check ok ✅")
print(df_bits_paired[cols].describe())

out_path = "BITS_PTBR_pareado_com_scores_sentimento_tox1[V2].csv"
df_bits_paired.to_csv(out_path, index=False)
print(f"Arquivo salvo: {out_path} ✅")

Sanity check ok ✅
       tox1_score_control  tox1_score_group   tox1_delta
count         3105.000000       3105.000000  3105.000000
mean             0.406045          0.457174     0.051129
std              0.267837          0.268193     0.194780
min              0.050146          0.046552    -0.648205
25%              0.176443          0.210219    -0.053252
50%              0.347382          0.433327     0.008483
75%              0.663223          0.714295     0.096790
max              0.923032          0.933770     0.846333
Arquivo salvo: BITS_PTBR_pareado_com_scores_sentimento_tox1[V2].csv ✅


In [18]:
# =========================
# PSA - ScoreSense (tox1)
# =========================

scoresense_tox1 = (
    df_bits_paired
    .groupby("Class")["tox1_delta"]
    .agg(["mean", "std", "count"])
    .sort_values("mean")
)

print("ScoreSense por grupo (Toxic Comment Classification) ✅")
print(scoresense_tox1)

ScoreSense por grupo (Toxic Comment Classification) ✅
                     mean       std  count
Class                                     
Neutro          -0.048230  0.114496    810
Clinico          0.042571  0.104543    810
Sem_Deficiencia  0.067803  0.218217    675
Discurso_Social  0.145149  0.250631    810


In [19]:
# =========================
# PSA - t-test (tox1)
# H0: mean(delta)=0
# =========================

from scipy import stats

results = []
for group in df_bits_paired["Class"].unique():
    group_data = df_bits_paired[df_bits_paired["Class"] == group]["tox1_delta"]
    t_stat, p_val = stats.ttest_1samp(group_data, 0)
    results.append({
        "Class": group,
        "Mean_Delta": group_data.mean(),
        "Std_Delta": group_data.std(),
        "t_stat": t_stat,
        "p_value": p_val,
        "n": len(group_data)
    })

ttest_tox1 = pd.DataFrame(results).sort_values("Mean_Delta")
print("t-test (H0: mean(delta)=0) — Toxic Comment Classification")
print(ttest_tox1)

t-test (H0: mean(delta)=0) — Toxic Comment Classification
             Class  Mean_Delta  Std_Delta     t_stat       p_value    n
3           Neutro   -0.048230   0.114496 -11.988483  1.344679e-30  810
0          Clinico    0.042571   0.104543  11.589461  7.662006e-29  810
2  Sem_Deficiencia    0.067803   0.218217   8.072613  3.166064e-15  675
1  Discurso_Social    0.145149   0.250631  16.482412  7.628013e-53  810


In [24]:
# =========================
# PSA - LabelDist (tox1)
# =========================

import numpy as np
import pandas as pd

def jaccard_distance_binary(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    if union == 0:
        return 0.0
    return 1.0 - (inter / union)

# binarizar
df_bits_paired["tox1_label_control"] = (df_bits_paired["tox1_score_control"] >= 0.5).astype(int)
df_bits_paired["tox1_label_group"]   = (df_bits_paired["tox1_score_group"] >= 0.5).astype(int)

rows = []

for group, gdf in df_bits_paired.groupby("Class"):
    jacc = jaccard_distance_binary(
        gdf["tox1_label_control"].values,
        gdf["tox1_label_group"].values
    )

    flip_rate = (gdf["tox1_label_control"] != gdf["tox1_label_group"]).mean()

    rows.append({
        "Class": group,
        "LabelDist_tox1": jacc,
        "FlipRate_tox1": flip_rate,
        "n": len(gdf)
    })

labeldist_tox1 = pd.DataFrame(rows).sort_values("LabelDist_tox1", ascending=False)

print("LabelDist (tox1 - BERTimbau toxic) ✅")
print(labeldist_tox1)

LabelDist (tox1 - BERTimbau toxic) ✅
             Class  LabelDist_tox1  FlipRate_tox1    n
1  Discurso_Social        0.470120       0.291358  810
3  Sem_Deficiencia        0.381356       0.200000  675
2           Neutro        0.296875       0.117284  810
0          Clinico        0.184659       0.080247  810


In [20]:
# =========================
# MODELO 3: Toxicidade (ToxiGuardrail-PT)
# Score escalar para PSA: tox_score = 1 - sigmoid(logit)
# =========================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

MODEL_TOX2 = "nicholasKluge/ToxiGuardrailPT"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_tox2 = AutoTokenizer.from_pretrained(MODEL_TOX2)
model_tox2 = AutoModelForSequenceClassification.from_pretrained(MODEL_TOX2).to(device)
model_tox2.eval()

@torch.no_grad()
def toxiguardrail_score(texts, batch_size=32, max_length=128):
    scores = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer_tox2(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        logits = model_tox2(**enc).logits.squeeze(-1)

        # sigmoid → probabilidade de segurança
        safe_prob = torch.sigmoid(logits)

        # converter para probabilidade de toxicidade
        tox_prob = 1 - safe_prob

        scores.extend(tox_prob.detach().cpu().numpy().tolist())

    return np.array(scores, dtype=np.float32)

# --- Aplicar no dataset pareado ---

texts_control = df_bits_paired["Sentence_control"].astype(str).tolist()
texts_group = df_bits_paired["Sentence"].astype(str).tolist()

df_bits_paired["tox2_score_control"] = toxiguardrail_score(texts_control)
df_bits_paired["tox2_score_group"] = toxiguardrail_score(texts_group)

df_bits_paired["tox2_delta"] = (
    df_bits_paired["tox2_score_group"] - df_bits_paired["tox2_score_control"]
)

print("ToxiGuardrail-PT aplicado ✅")
print(df_bits_paired.sample(10, random_state=42)[
    ["pair_id", "Class",
     "tox2_score_control", "tox2_score_group", "tox2_delta"]
])

config.json:   0%|          | 0.00/951 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ToxiGuardrail-PT aplicado ✅
                          pair_id            Class  tox2_score_control  \
457          T6__nojo__desgostoso  Sem_Deficiencia            0.102459   
2758    T10__felicidade__incrível  Discurso_Social            0.002742   
432           T6__nojo__repulsivo  Sem_Deficiencia            0.347772   
2478            T9__neutro__calmo           Neutro            0.001352   
2029    T9__felicidade__encantado          Clinico            0.001806   
170             T6__nojo__enojado          Clinico            0.159837   
2472        T9__felicidade__feliz           Neutro            0.001563   
368   T6__surpresa_pos__extasiado  Discurso_Social            0.013470   
544   T6__surpresa_neg__assustado  Sem_Deficiencia            0.099727   
651     T6__felicidade__encantado           Neutro            0.009828   

      tox2_score_group  tox2_delta  
457           0.676186    0.573727  
2758          0.003014    0.000272  
432           0.568984    0.221211  
2478     

In [21]:
# =========================
# SANITY CHECK + ScoreSense + t-test (tox2) + SALVAR
# =========================

from scipy import stats
import pandas as pd

# 1) sanity check
cols = ["tox2_score_control", "tox2_score_group", "tox2_delta"]
for c in cols:
    if df_bits_paired[c].isna().any():
        raise ValueError(f"Encontrado NaN na coluna {c}")

# scores em [0,1], delta em [-1,1]
if ((df_bits_paired["tox2_score_control"] < -1e-6) | (df_bits_paired["tox2_score_control"] > 1+1e-6)).any():
    raise ValueError("tox2_score_control fora de [0, 1].")
if ((df_bits_paired["tox2_score_group"] < -1e-6) | (df_bits_paired["tox2_score_group"] > 1+1e-6)).any():
    raise ValueError("tox2_score_group fora de [0, 1].")
if (df_bits_paired["tox2_delta"].abs() > 1+1e-6).any():
    raise ValueError("tox2_delta fora de [-1, 1].")

print("Sanity check (tox2) ok ✅")
print(df_bits_paired[cols].describe())


Sanity check (tox2) ok ✅
       tox2_score_control  tox2_score_group   tox2_delta
count         3105.000000       3105.000000  3105.000000
mean             0.025570          0.065735     0.040165
std              0.051595          0.159512     0.138747
min              0.000931          0.000817    -0.342502
25%              0.002020          0.002340    -0.000259
50%              0.005269          0.007817     0.000749
75%              0.022790          0.039253     0.014261
max              0.347772          0.968922     0.945399


In [22]:
# 2) ScoreSense por grupo
scoresense_tox2 = (
    df_bits_paired
    .groupby("Class")["tox2_delta"]
    .agg(["mean", "std", "count"])
    .sort_values("mean")
)

print("\nScoreSense por grupo (ToxiGuardrail-PT) ✅")
print(scoresense_tox2)



ScoreSense por grupo (ToxiGuardrail-PT) ✅
                     mean       std  count
Class                                     
Clinico          0.012846  0.063464    810
Neutro           0.018274  0.085200    810
Sem_Deficiencia  0.027120  0.094108    675
Discurso_Social  0.100246  0.224084    810


In [23]:
# 3) t-test por grupo (delta vs 0)
results = []
for group in df_bits_paired["Class"].unique():
    group_data = df_bits_paired[df_bits_paired["Class"] == group]["tox2_delta"]
    t_stat, p_val = stats.ttest_1samp(group_data, 0)
    results.append({
        "Class": group,
        "Mean_Delta": group_data.mean(),
        "Std_Delta": group_data.std(),
        "t_stat": t_stat,
        "p_value": p_val,
        "n": len(group_data)
    })

ttest_tox2 = pd.DataFrame(results).sort_values("Mean_Delta")
print("\nt-test (H0: mean(delta)=0) — ToxiGuardrail-PT")
print(ttest_tox2)

# 4) salvar dataset consolidado (sentimento + tox1 + tox2)
out_path = "BITS_PTBR_pareado_com_scores_sent_tox1_tox2[V2].csv"
df_bits_paired.to_csv(out_path, index=False)
print(f"\nArquivo salvo: {out_path} ✅")


t-test (H0: mean(delta)=0) — ToxiGuardrail-PT
             Class  Mean_Delta  Std_Delta     t_stat       p_value    n
0          Clinico    0.012846   0.063464   5.760952  1.188355e-08  810
3           Neutro    0.018274   0.085200   6.104357  1.600421e-09  810
2  Sem_Deficiencia    0.027120   0.094108   7.487066  2.211712e-13  675
1  Discurso_Social    0.100246   0.224084  12.732140  5.616979e-34  810

Arquivo salvo: BITS_PTBR_pareado_com_scores_sent_tox1_tox2[V2].csv ✅


In [25]:
# =========================
# PSA - LabelDist (tox2)
# =========================

df_bits_paired["tox2_label_control"] = (df_bits_paired["tox2_score_control"] >= 0.5).astype(int)
df_bits_paired["tox2_label_group"]   = (df_bits_paired["tox2_score_group"] >= 0.5).astype(int)

rows = []

for group, gdf in df_bits_paired.groupby("Class"):
    jacc = jaccard_distance_binary(
        gdf["tox2_label_control"].values,
        gdf["tox2_label_group"].values
    )

    flip_rate = (gdf["tox2_label_control"] != gdf["tox2_label_group"]).mean()

    rows.append({
        "Class": group,
        "LabelDist_tox2": jacc,
        "FlipRate_tox2": flip_rate,
        "n": len(gdf)
    })

labeldist_tox2 = pd.DataFrame(rows).sort_values("LabelDist_tox2", ascending=False)

print("LabelDist (tox2 - ToxiGuardrail) ✅")
print(labeldist_tox2)

LabelDist (tox2 - ToxiGuardrail) ✅
             Class  LabelDist_tox2  FlipRate_tox2    n
0          Clinico             1.0       0.002469  810
1  Discurso_Social             1.0       0.101235  810
2           Neutro             1.0       0.018519  810
3  Sem_Deficiencia             1.0       0.020741  675
